#Enhancement done for Explainability

#1. Batched, reusable occlusion sensitivity

What: Replaced the original single-image, hardcoded occlusion heatmap with `explain_prediction()` — runs all 49 grid cells through the backbone in batches instead of one forward pass per cell, and works on any image path.

Why: The original version was slow (49 individual forward passes) and could only ever explain one fixed validation example, which isn't usable for a live app or for systematic diagnosis.

Result: ~25x fewer backbone calls per explanation, and the same function now powers both the notebook diagnostics and the eventual Day 5 live-upload path.


#2. Nearest-catalog-image lookup

What: Added `EmbeddingIndex` (loaded from Day 1's saved gallery) and `find_similar_products()`, wired into `explain_prediction()` and `plot_explanation()` so each explanation now shows the top-k visually similar catalog products alongside the heatmap.

Why: to have explainability goal of "similar catalog images and Grad-CAM evidence"

Result: Every explanation panel now answers two different questions side by side: "which pixels mattered" (heatmap) and "what does the model think this looks like" (nearest neighbors).

#3. Grad-CAM after recognition classifier fine tuning

What: Added `YOLOBoxScoreTarget` and `yolo_gradcam()`, running Grad-CAM against `detector.model`'s last conv block.

Why: additional confidence applied after recongition / detection stage

Result: Detection-stage explainability ("why did it box something here") now exists as a separate, complementary view to Day 3's recognition-stage explainability ("why did it call this a Granny Smith apple").

#4. Background-shortcut diagnostic

What: Added a loop checking `explain_prediction()` across 5 examples each of a misread's true class and predicted class.

Why: A misclassification with a high confidence with the heatmap on the background, not the product, suggested possible shortcut learning rather than genuine uncertainty.

Result: Direct evidence of whether the background attention is class-specific and systematic, or an isolated case — informs whether retraining is actually justified.

#5. Crop-aware retraining experiment

What: Added a second head (`head_crop_aware`) trained on Day 1's clean photos combined with Day 2's real detector crops, plus a before/after comparison of the occlusion explanation on the same problem image.

Why: The original head only ever saw clean, well-framed catalog photos — never the tighter, sometimes-imperfect crops it will actually receive at inference from Day 2's detector.

Result: Direct accuracy and heatmap-behavior comparison between clean-only and crop-aware training, to test whether exposure to real crop framing reduces reliance on background cues.

#6. Crop-aware training preserved

What: Modified the retraining cell to load Day 2's crops_manifest.csv and merge it into base_pool before computing features, instead of rebuilding head_before from clean photos alone.

Why: Day 4 was retraining entirely from scratch using list_images(root) — the same clean-only images that caused the fridge-background shortcut in Day 3. Since Day 5 always prefers head_v2.pt over head.pt, this meant Day 3's crop-aware fix would get silently discarded the moment Day 4 ran, regardless of how well it worked.

Result: head_before and head_v2 now both build on the same crop-aware foundation Day 3 established, so the improvement actually survives into the shipped model instead of being reset.
